# Zenith AI — Kaggle Training

Trains the ~75M-parameter `configs/zenith_kaggle.yaml` variant of Zenith on a Kaggle GPU (T4/P100).

**Before running:** in the notebook side panel, set **Accelerator = GPU** and **Internet = On** (needed for `git clone` and the TinyStories dataset download).

**Session limit:** Kaggle sessions cap at ~12h. Training is checkpointed every 500 steps to `/kaggle/working/checkpoints` and auto-resumes from `latest.pt` if present, so if a run gets cut off, save `/kaggle/working/checkpoints` as a Kaggle Dataset, attach it as input on the next run, copy it back into `checkpoints/`, and re-run this notebook — it will pick up where it left off.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone the repo
!rm -rf /kaggle/working/Zenith-AI
!git clone --depth 1 https://github.com/ItzAditya43/Zenith-AI.git /kaggle/working/Zenith-AI
%cd /kaggle/working/Zenith-AI

In [ ]:
# Kaggle images already ship torch with CUDA; only install the extra deps
!pip install -q tokenizers datasets pyyaml tqdm

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## Optional: resume from a previous session

If you attached a Kaggle Dataset containing a prior `checkpoints/` output, copy it in before training starts. Otherwise skip this cell — training starts fresh.

In [ ]:
# import shutil, os
# prev_ckpt_dataset = "/kaggle/input/<your-attached-checkpoint-dataset>"
# os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
# for f in os.listdir(prev_ckpt_dataset):
#     shutil.copy(os.path.join(prev_ckpt_dataset, f), "/kaggle/working/checkpoints/")
# print("Restored checkpoints:", os.listdir("/kaggle/working/checkpoints"))

## Data pipeline

Downloads TinyStories, trains an 8192-vocab BPE tokenizer from scratch, then tokenizes and packs it into fixed-length shards. Skipped automatically on resume if the packed `.npy` files already exist.

In [ ]:
import os
if not os.path.exists("data/train.txt"):
    !python -m zenith.data.prepare dump --output-dir data

In [ ]:
if not os.path.exists("data/tokenizer.json"):
    !python -m zenith.tokenizer.train_tokenizer --input data/train.txt --output data/tokenizer.json --vocab-size 8192

In [ ]:
# Full TinyStories corpus this time (Kaggle has the disk + GPU headroom for it).
# Context length is 512 here (vs. 256 in the local laptop config) — must repack even if you ran the smaller config before.
if not os.path.exists("data/train_packed.npy"):
    !python -m zenith.data.prepare pack --input data/train.txt --tokenizer data/tokenizer.json --output data/train_packed.npy --seq-len 512
if not os.path.exists("data/val_packed.npy"):
    !python -m zenith.data.prepare pack --input data/val.txt --tokenizer data/tokenizer.json --output data/val_packed.npy --seq-len 512

## Train

~75.5M params. Checkpoints land in `/kaggle/working/checkpoints/latest.pt` (auto-resumed if this cell reruns) and are included in the notebook's persistent output automatically.

In [ ]:
!python -m zenith.training.train configs/zenith_kaggle.yaml

## Try it

In [ ]:
from zenith.cli import load_model_and_tokenizer
from zenith.inference.generate import generate_text

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tok, cfg = load_model_and_tokenizer("/kaggle/working/checkpoints/latest.pt", "data/tokenizer.json", device)

for prompt in ["Once upon a time", "The little dog was very", "Tom and Lily went to the park"]:
    text = generate_text(model, tok, prompt, max_new_tokens=100, temperature=0.7, top_k=40, device=device)
    print("PROMPT:", prompt)
    print("OUTPUT:", prompt + text)
    print()

## Save checkpoints as a Dataset (for resuming later)

Kaggle keeps `/kaggle/working` as notebook output automatically when you **Save Version**. To resume in a future session: after saving this version, go to "New Dataset" → source it from this notebook's output → attach that dataset as input next time → uncomment the restore cell above.